In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# CoroNet: The Master Classification Pipeline (Part 1 + Part 2)
# ==========================================
import os
import glob
import re
import gc
import numpy as np
import numpy.linalg as npl
import nibabel as nib
from nibabel.affines import apply_affine
from scipy.ndimage import label, distance_transform_edt
from skimage.morphology import skeletonize

# 1. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
MAMBA_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Coronary_Mapping_Input")
CHAMBER_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Helper/Totalsegmentor_Heart")
OUT_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Helper/Classified_Trees_Centerline")

# Create the output directory to hold the 20 processed patients
os.makedirs(OUT_DIR, exist_ok=True)

# 2. Dynamic Patient Discovery
print("🔍 Scanning folders for patients...")
mamba_files = glob.glob(os.path.join(MAMBA_DIR, "ICC_Patient_*.nii.gz"))

patient_ids = []
for f in mamba_files:
    match = re.search(r'ICC_Patient_(\d+)\.nii\.gz', os.path.basename(f))
    if match:
        patient_ids.append(match.group(1))

patient_ids = sorted(patient_ids)
print(f"📊 Found {len(patient_ids)} total patients in the dataset.")

# ==========================================
# THE BATCH LOOP
# ==========================================
for pid in patient_ids:
    print(f"\n======================================")
    print(f"⚙️ Processing Patient {pid}...")

    final_save_path = os.path.join(OUT_DIR, f"classified_coronary_tree_{pid}.nii.gz")

    # ⏩ Smart Skipping: If the file already exists, jump to the next patient
    if os.path.exists(final_save_path):
        print(f"   ⏩ Tree already exists. Skipping Patient {pid}!")
        continue

    try:
        # --- PART 1: THE ANATOMICAL SETUP ---

        # A. Load U-Mamba Image and Skeletonize
        print(f"   -> Loading U-Mamba mask and skeletonizing...")
        artery_img = nib.load(os.path.join(MAMBA_DIR, f"ICC_Patient_{pid}.nii.gz"))
        artery_data = artery_img.get_fdata()
        canvas_shape = artery_img.shape

        skeleton_data = skeletonize((artery_data > 0).astype(np.uint8))

        # B. Load TotalSegmentator Chambers and create a JOINT Bounding Box
        print(f"   -> Loading TotalSegmentator chambers...")
        chamber_img = nib.load(os.path.join(CHAMBER_DIR, f"ICC_Patient_{pid}", "heart.nii.gz"))
        mask_data = chamber_img.get_fdata()
        chamber_inv_affine = npl.inv(chamber_img.affine)

        # Map skeleton coordinates to the chamber's voxel space
        skel_coords_artery = np.array(np.nonzero(skeleton_data)).T  # (N, 3)
        coords_heart = np.array(np.nonzero(mask_data)).T  # (M, 3)

        if coords_heart.size == 0 or skel_coords_artery.size == 0:
            print("   🚨 Missing heart or skeleton data. Skipping.")
            continue

        # Convert artery voxels -> physical mm -> chamber voxels
        skel_coords_mm = apply_affine(artery_img.affine, skel_coords_artery)
        skel_coords_chamber = apply_affine(chamber_inv_affine, skel_coords_mm)
        skel_coords_chamber = np.round(skel_coords_chamber).astype(int)

        # Get bounds for both heart and mapped skeleton in CHAMBER space
        min_x = min(coords_heart[:, 0].min(), skel_coords_chamber[:, 0].min())
        min_y = min(coords_heart[:, 1].min(), skel_coords_chamber[:, 1].min())
        min_z = min(coords_heart[:, 2].min(), skel_coords_chamber[:, 2].min())

        max_x = max(coords_heart[:, 0].max(), skel_coords_chamber[:, 0].max())
        max_y = max(coords_heart[:, 1].max(), skel_coords_chamber[:, 1].max())
        max_z = max(coords_heart[:, 2].max(), skel_coords_chamber[:, 2].max())

        # Pad the box slightly so we don't cut off any edges
        PADDING = 20
        shape = mask_data.shape
        min_x, max_x = max(0, min_x - PADDING), min(shape[0], max_x + PADDING)
        min_y, max_y = max(0, min_y - PADDING), min(shape[1], max_y + PADDING)
        min_z, max_z = max(0, min_z - PADDING), min(shape[2], max_z + PADDING)

        # Crop the raw chamber data to save massive amounts of RAM
        chamber_crop = mask_data[min_x:max_x, min_y:max_y, min_z:max_z]

        # C. Calculate Fresh Distance Maps ONLY inside the cropped box
        print("   -> Calculating Euclidean Distances (Memory Safe Mode)...")
        # LA = Label 2, LV = Label 3, RA = Label 4
        # Fix: If a chamber is completely missing, its distance is infinity, not 0.
        d_LA_crop = distance_transform_edt(chamber_crop != 2) if np.any(chamber_crop == 2) else np.full(chamber_crop.shape, np.inf)
        d_LV_crop = distance_transform_edt(chamber_crop != 3) if np.any(chamber_crop == 3) else np.full(chamber_crop.shape, np.inf)
        d_RA_crop = distance_transform_edt(chamber_crop != 4) if np.any(chamber_crop == 4) else np.full(chamber_crop.shape, np.inf)


        # --- PART 2: THE CLASSIFICATION LOGIC ---

        # Create the blank 3D canvas for the final tree in original ARTERY space
        classified_skeleton = np.zeros(canvas_shape, dtype=np.uint8)

        # D. Group the 1-pixel SKELETON into continuous branches
        print("   -> Grouping skeleton into continuous branches...")
        structure = np.ones((3, 3, 3))
        labeled_skeleton, num_branches = label(skeleton_data > 0, structure=structure)

        print(f"   -> Found {num_branches} branches. Running Pure Anchors vote...")
        rca_count = 0
        left_count = 0

        # E. Hold the Majority Vote on the ENTIRE branch
        for branch_id in range(1, num_branches + 1):
            branch_coords_artery = np.argwhere(labeled_skeleton == branch_id)

            # Map this specific branch to chamber space for voting
            branch_coords_mm = apply_affine(artery_img.affine, branch_coords_artery)
            branch_coords_chamber = apply_affine(chamber_inv_affine, branch_coords_mm)
            branch_coords_chamber = np.round(branch_coords_chamber).astype(int)

            rca_votes, left_votes = 0, 0

            for i in range(len(branch_coords_chamber)):
                # Translate to the cropped box coordinates
                cx = branch_coords_chamber[i, 0] - min_x
                cy = branch_coords_chamber[i, 1] - min_y
                cz = branch_coords_chamber[i, 2] - min_z

                # Safety check just in case
                if 0 <= cx < chamber_crop.shape[0] and 0 <= cy < chamber_crop.shape[1] and 0 <= cz < chamber_crop.shape[2]:
                    right_score = d_RA_crop[cx, cy, cz]
                    left_score = min(d_LA_crop[cx, cy, cz],
                                     d_LV_crop[cx, cy, cz])

                    if right_score < left_score:
                        rca_votes += 1
                    else:
                        left_votes += 1

            # The winner takes the whole branch
            winner_color = 1 if rca_votes > left_votes else 2

            if winner_color == 1:
                rca_count += len(branch_coords_artery)
            else:
                left_count += len(branch_coords_artery)

            # Paint the canvas using original artery coordinates
            for x, y, z in branch_coords_artery:
                classified_skeleton[x, y, z] = winner_color

        print(f"   ✓ Classification complete! RCA pixels: {rca_count} | Left System: {left_count}")

        # F. Save the final mapped NIfTI file
        final_img = nib.Nifti1Image(classified_skeleton, artery_img.affine)
        nib.save(final_img, final_save_path)
        print(f"   💾 Saved to: {final_save_path}")

    except Exception as e:
        print(f"   🚨 ERROR processing Patient {pid}: {str(e)}")
        print("   -> Pipeline is keeping the loop alive and moving to the next patient.")

    # --- THE RAM VACUUM ---
    # Wipe the heavy 3D arrays from memory before starting the next patient
    try:
        del artery_data, mask_data, chamber_crop
        del d_LA_crop, d_LV_crop, d_RA_crop
        del skeleton_data, classified_skeleton, labeled_skeleton
        del skel_coords_artery, coords_heart, skel_coords_chamber, skel_coords_mm
    except NameError:
        pass # Ignores if an error caused variables to not be created

    gc.collect()

print("\n🎉 PHASE 3 BATCH PROCESSING COMPLETE!")

🔍 Scanning folders for patients...
📊 Found 20 total patients in the dataset.

⚙️ Processing Patient 0169...
   -> Loading U-Mamba mask and skeletonizing...
   -> Loading TotalSegmentator chambers...
   -> Calculating Euclidean Distances (Memory Safe Mode)...
   -> Grouping skeleton into continuous branches...
   -> Found 3 branches. Running Pure Anchors vote...
   ✓ Classification complete! RCA pixels: 525 | Left System: 718
   💾 Saved to: /content/drive/MyDrive/COROnet_Datasets/Segmentation/Classified_Trees_V3/classified_coronary_tree_0169.nii.gz

⚙️ Processing Patient 0170...
   -> Loading U-Mamba mask and skeletonizing...
   -> Loading TotalSegmentator chambers...
   -> Calculating Euclidean Distances (Memory Safe Mode)...
   -> Grouping skeleton into continuous branches...
   -> Found 8 branches. Running Pure Anchors vote...
   ✓ Classification complete! RCA pixels: 461 | Left System: 690
   💾 Saved to: /content/drive/MyDrive/COROnet_Datasets/Segmentation/Classified_Trees_V3/classi

In [ ]:
# ==========================================
# CoroNet Utility: Batch 3D Skeleton Inflater
# ==========================================
import os
import glob
import re
import gc
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_dilation

# 1. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
TREES_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Helper/Classified_Trees_Centerline")
OUT_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Helper/Classified_Trees_Thickened")

# Create the output folder so we don't mix thick and thin files!
os.makedirs(OUT_DIR, exist_ok=True)

# 2. Find all your processed 1-pixel trees
print("🔍 Scanning for 1-pixel classified trees...")
tree_files = glob.glob(os.path.join(TREES_DIR, "classified_coronary_tree_*.nii.gz"))

patient_ids = []
for f in tree_files:
    match = re.search(r'classified_coronary_tree_(\d+)\.nii\.gz', os.path.basename(f))
    if match:
        patient_ids.append(match.group(1))

patient_ids = sorted(patient_ids)
print(f"📊 Found {len(patient_ids)} patients to inflate.")

# ==========================================
# THE BATCH INFLATION LOOP
# ==========================================
for pid in patient_ids:
    print(f"\n======================================")
    print(f"🎈 Inflating Patient {pid}...")

    final_save_path = os.path.join(OUT_DIR, f"THICK_tree_{pid}.nii.gz")

    # Smart Skipping
    if os.path.exists(final_save_path):
        print(f"   ⏩ Thick file already exists. Skipping Patient {pid}!")
        continue

    try:
        # Load the 1-pixel data
        img = nib.load(os.path.join(TREES_DIR, f"classified_coronary_tree_{pid}.nii.gz"))
        data = img.get_fdata()

        # Inflate the 1-pixel wire into a 3-pixel pipe (iterations=2)
        print("   -> Dilating RCA (Label 1)...")
        rca_thick = binary_dilation(data == 1, iterations=2)

        print("   -> Dilating Left System (Label 2)...")
        left_thick = binary_dilation(data == 2, iterations=2)

        # Create a new canvas and paint the thick pipes
        thick_canvas = np.zeros_like(data, dtype=np.uint8)
        thick_canvas[rca_thick] = 1
        # Ensure Left System doesn't overwrite RCA where they touch at the root!
        thick_canvas[left_thick & ~rca_thick] = 2

        # Save the new THICK version
        nib.save(nib.Nifti1Image(thick_canvas, img.affine), final_save_path)
        print(f"   💾 Saved beautiful 3D view to: THICK_tree_{pid}.nii.gz")

    except Exception as e:
        print(f"   🚨 ERROR inflating Patient {pid}: {str(e)}")

    # The RAM Vacuum
    try:
        del data, rca_thick, left_thick, thick_canvas
    except NameError:
        pass
    gc.collect()

print("\n🎉 ALL TREES THICKENED! Ready for ITK-SNAP Inspection.")

🔍 Scanning for 1-pixel classified trees...
📊 Found 20 patients to inflate.

🎈 Inflating Patient 0169...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_tree_0169.nii.gz

🎈 Inflating Patient 0170...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_tree_0170.nii.gz

🎈 Inflating Patient 0171...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_tree_0171.nii.gz

🎈 Inflating Patient 0172...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_tree_0172.nii.gz

🎈 Inflating Patient 0173...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_tree_0173.nii.gz

🎈 Inflating Patient 0174...
   -> Dilating RCA (Label 1)...
   -> Dilating Left System (Label 2)...
   💾 Saved beautiful 3D view to: THICK_

In [ ]:
import os
import glob
import re
import gc
import numpy as np
import numpy.linalg as npl
import nibabel as nib
from nibabel.affines import apply_affine
from scipy.ndimage import distance_transform_edt, label
from scipy.spatial import cKDTree

# 1. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
TREES_DIR = '/content/drive/MyDrive/COROnet_Drive/Segmentation/Classified_Trees_Thickened'
CHAMBER_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Totalsegmentor_Heart")
OUT_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Classified_Trees_Centerline_Full")

os.makedirs(OUT_DIR, exist_ok=True)

# --- Configuration Parameters ---
# Adjust this value (in mm) to bias the provisional classification towards LCx.
LCX_BIAS_MM = 5.0
# Adjust this value (in mm) to bias the final core anchoring towards LAD (helps keep Left Main as LAD).
LAD_ANCHOR_BIAS_MM = 15.0
# --------------------------------

# 2. Dynamic Patient Discovery
print("🔍 Scanning for V3 2-class trees...")
tree_files = glob.glob(os.path.join(TREES_DIR, "classified_coronary_tree_*.nii.gz"))

patient_ids = []
for f in tree_files:
    match = re.search(r'classified_coronary_tree_(\d+)\.nii\.gz', os.path.basename(f))
    if match:
        patient_ids.append(match.group(1))

patient_ids = sorted(patient_ids)
print(f"📊 Found {len(patient_ids)} patients for LAD/LCx splitting.")

# ==========================================
# THE BATCH LOOP
# ==========================================
for pid in patient_ids:
    print(f"\n======================================")
    print(f"⚙️ Splitting Left System for Patient {pid}...")

    final_save_path = os.path.join(OUT_DIR, f"full_classified_tree_{pid}.nii.gz")

    if os.path.exists(final_save_path):
        print(f"   ⏩ Full tree already exists. Skipping Patient {pid}!")
        continue

    try:
        # A. Load the 2-class tree (1=RCA, 2=Left System)
        tree_img = nib.load(os.path.join(TREES_DIR, f"classified_coronary_tree_{pid}.nii.gz"))
        tree_data = tree_img.get_fdata()

        # B. Load TotalSegmentator chambers
        chamber_img = nib.load(os.path.join(CHAMBER_DIR, f"ICC_Patient_{pid}", "heart.nii.gz"))
        mask_data = chamber_img.get_fdata()
        chamber_inv_affine = npl.inv(chamber_img.affine)

        # Isolate Left System pixels
        left_coords_artery = np.array(np.nonzero(tree_data == 2)).T
        coords_heart = np.array(np.nonzero(mask_data)).T

        if coords_heart.size == 0 or left_coords_artery.size == 0:
            print("   🚨 Missing heart or Left System data. Skipping.")
            continue

        # C. Map Left System coordinates to the chamber's voxel space
        left_coords_mm = apply_affine(tree_img.affine, left_coords_artery)
        left_coords_chamber = apply_affine(chamber_inv_affine, left_coords_mm)
        left_coords_chamber = np.round(left_coords_chamber).astype(int)

        # Bounding box logic for efficiency
        min_x = min(coords_heart[:, 0].min(), left_coords_chamber[:, 0].min())
        min_y = min(coords_heart[:, 1].min(), left_coords_chamber[:, 1].min())
        min_z = min(coords_heart[:, 2].min(), left_coords_chamber[:, 2].min())

        max_x = max(coords_heart[:, 0].max(), left_coords_chamber[:, 0].max())
        max_y = max(coords_heart[:, 1].max(), left_coords_chamber[:, 1].max())
        max_z = max(coords_heart[:, 2].max(), left_coords_chamber[:, 2].max())

        PADDING = 20
        shape = mask_data.shape
        min_x, max_x = max(0, min_x - PADDING), min(shape[0], max_x + PADDING)
        min_y, max_y = max(0, min_y - PADDING), min(shape[1], max_y + PADDING)
        min_z, max_z = max(0, min_z - PADDING), min(shape[2], max_z + PADDING)

        chamber_crop = mask_data[min_x:max_x, min_y:max_y, min_z:max_z]

        # D. Calculate Distances to LA (Label 2), RA (Label 4), and RV (Label 5)
        print("   -> Calculating distance maps for provisional LAD vs LCx classification...")
        d_LA_crop = distance_transform_edt(chamber_crop != 2) if np.any(chamber_crop == 2) else np.full(chamber_crop.shape, np.inf)
        d_RA_crop = distance_transform_edt(chamber_crop != 4) if np.any(chamber_crop == 4) else np.full(chamber_crop.shape, np.inf)
        d_RV_crop = distance_transform_edt(chamber_crop != 5) if np.any(chamber_crop == 5) else np.full(chamber_crop.shape, np.inf)

        # E. Provisional Pixel-by-Pixel Classification
        print("   -> Running provisional pixel-by-pixel classification...")
        new_classified_tree = np.zeros_like(tree_data, dtype=np.uint8)
        new_classified_tree[tree_data == 1] = 1  # Preserve RCA

        provisional_canvas = np.zeros_like(tree_data, dtype=np.uint8)

        for i in range(len(left_coords_chamber)):
            cx = left_coords_chamber[i, 0] - min_x
            cy = left_coords_chamber[i, 1] - min_y
            cz = left_coords_chamber[i, 2] - min_z

            ax, ay, az = left_coords_artery[i]

            if 0 <= cx < chamber_crop.shape[0] and 0 <= cy < chamber_crop.shape[1] and 0 <= cz < chamber_crop.shape[2]:
                la_score = d_LA_crop[cx, cy, cz]
                anterior_score = min(d_RA_crop[cx, cy, cz], d_RV_crop[cx, cy, cz])

                if la_score < (anterior_score - LCX_BIAS_MM):
                    provisional_canvas[ax, ay, az] = 3  # LCx
                else:
                    provisional_canvas[ax, ay, az] = 2  # LAD
            else:
                provisional_canvas[ax, ay, az] = 2  # Fallback LAD

        # F. Core Anchoring (Reassigning tips to the nearest Main Core)
        print("   -> Anchoring fragmented tips to the Main LAD and Main LCx Cores...")
        structure = np.ones((3, 3, 3))
        lad_labeled, _ = label(provisional_canvas == 2, structure=structure)
        lcx_labeled, _ = label(provisional_canvas == 3, structure=structure)

        # Identify the largest connected component for each class (the "Core")
        lad_counts = np.bincount(lad_labeled.ravel())
        lad_counts[0] = 0  # Ignore background
        main_lad_id = lad_counts.argmax() if len(lad_counts) > 1 else 0

        lcx_counts = np.bincount(lcx_labeled.ravel())
        lcx_counts[0] = 0  # Ignore background
        main_lcx_id = lcx_counts.argmax() if len(lcx_counts) > 1 else 0

        main_lad_coords = np.argwhere(lad_labeled == main_lad_id)
        main_lcx_coords = np.argwhere(lcx_labeled == main_lcx_id)

        lad_count = 0
        lcx_count = 0

        if len(main_lad_coords) > 0 and len(main_lcx_coords) > 0:
            # Convert core coordinates to MM space for accurate distance calculations
            main_lad_mm = apply_affine(tree_img.affine, main_lad_coords)
            main_lcx_mm = apply_affine(tree_img.affine, main_lcx_coords)

            lad_tree = cKDTree(main_lad_mm)
            lcx_tree = cKDTree(main_lcx_mm)

            for i in range(len(left_coords_mm)):
                point = left_coords_mm[i]
                dist_to_lad, _ = lad_tree.query(point)
                dist_to_lcx, _ = lcx_tree.query(point)

                # Re-assign based on distance to the massive core bodies
                # Bias applied here to favor LAD for the Left Main root
                if dist_to_lcx < (dist_to_lad - LAD_ANCHOR_BIAS_MM):
                    final_label = 3
                    lcx_count += 1
                else:
                    final_label = 2
                    lad_count += 1

                ax, ay, az = left_coords_artery[i]
                new_classified_tree[ax, ay, az] = final_label
        else:
            # Fallback if a core completely failed to generate (extremely rare)
            print("   ⚠️ Warning: Missing a distinct core. Falling back to provisional labels.")
            for i in range(len(left_coords_artery)):
                ax, ay, az = left_coords_artery[i]
                final_label = provisional_canvas[ax, ay, az]
                new_classified_tree[ax, ay, az] = final_label
                if final_label == 2:
                    lad_count += 1
                else:
                    lcx_count += 1

        print(f"   ✓ Anchoring complete! LAD pixels: {lad_count} | LCx pixels: {lcx_count}")

        # G. Save the 3-class NIfTI file
        final_img = nib.Nifti1Image(new_classified_tree, tree_img.affine)
        nib.save(final_img, final_save_path)
        print(f"   💾 Saved to: {final_save_path}")

    except Exception as e:
        print(f"   🚨 ERROR processing Patient {pid}: {str(e)}")

    # RAM Vacuum
    try:
        del tree_data, mask_data, chamber_crop, provisional_canvas
        del d_LA_crop, d_RA_crop, d_RV_crop
        del lad_labeled, lcx_labeled, new_classified_tree
        del left_coords_artery, coords_heart, left_coords_chamber, left_coords_mm
    except NameError:
        pass

    gc.collect()

print("\n🎉 LEFT SYSTEM BIFURCATION SPLIT COMPLETE!")

🔍 Scanning for V3 2-class trees...
📊 Found 20 patients for LAD/LCx splitting.

⚙️ Splitting Left System for Patient 0169...
   -> Calculating distance maps for provisional LAD vs LCx classification...
   -> Running provisional pixel-by-pixel classification...
   -> Anchoring fragmented tips to the Main LAD and Main LCx Cores...
   ✓ Anchoring complete! LAD pixels: 342 | LCx pixels: 376
   💾 Saved to: /content/drive/MyDrive/COROnet_Datasets/Segmentation/Classified_Trees_V8_Full/full_classified_tree_0169.nii.gz

⚙️ Splitting Left System for Patient 0170...
   -> Calculating distance maps for provisional LAD vs LCx classification...
   -> Running provisional pixel-by-pixel classification...
   -> Anchoring fragmented tips to the Main LAD and Main LCx Cores...
   ✓ Anchoring complete! LAD pixels: 443 | LCx pixels: 247
   💾 Saved to: /content/drive/MyDrive/COROnet_Datasets/Segmentation/Classified_Trees_V8_Full/full_classified_tree_0170.nii.gz

⚙️ Splitting Left System for Patient 0171...
  

In [ ]:
# ==========================================
# CoroNet Utility: Batch 3D Skeleton Inflater (3-Class V5)
# ==========================================
import os
import glob
import re
import gc
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_dilation

# 1. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
TREES_DIR_V5 = os.path.join(DRIVE_ROOT, "Segmentation/Classified_Trees_Centerline_Full")
OUT_DIR_THICK = os.path.join(DRIVE_ROOT, "Segmentation/Classified_Trees_Thickened_Full")

# Create the output folder
os.makedirs(OUT_DIR_THICK, exist_ok=True)

# 2. Find all processed 3-class trees
print("🔍 Scanning for V5 3-class classified trees...")
tree_files = glob.glob(os.path.join(TREES_DIR_V5, "full_classified_tree_*.nii.gz"))

patient_ids = []
for f in tree_files:
    match = re.search(r'full_classified_tree_(\d+)\.nii\.gz', os.path.basename(f))
    if match:
        patient_ids.append(match.group(1))

patient_ids = sorted(patient_ids)
print(f"📊 Found {len(patient_ids)} patients to inflate.")

# ==========================================
# THE BATCH INFLATION LOOP
# ==========================================
for pid in patient_ids:
    print(f"\n======================================")
    print(f"🎈 Inflating Patient {pid}...")

    final_save_path = os.path.join(OUT_DIR_THICK, f"ICC_Patient_Seg_{pid}.nii.gz")

    # Smart Skipping
    if os.path.exists(final_save_path):
        print(f"   ⏩ Thick file already exists. Skipping Patient {pid}!")
        continue

    try:
        # Load the 1-pixel data
        img = nib.load(os.path.join(TREES_DIR_V5, f"full_classified_tree_{pid}.nii.gz"))
        data = img.get_fdata()

        # Inflate the 1-pixel wire into a 3-pixel pipe (iterations=2)
        print("   -> Dilating RCA (Label 1)...")
        rca_thick = binary_dilation(data == 1, iterations=2)

        print("   -> Dilating LAD (Label 2)...")
        lad_thick = binary_dilation(data == 2, iterations=2)

        print("   -> Dilating LCx (Label 3)...")
        lcx_thick = binary_dilation(data == 3, iterations=2)

        # Create a new canvas and paint the thick pipes
        thick_canvas = np.zeros_like(data, dtype=np.uint8)

        # Assign colors with priority to prevent overwriting at the roots
        thick_canvas[rca_thick] = 1
        thick_canvas[lad_thick & ~rca_thick] = 2
        thick_canvas[lcx_thick & ~rca_thick & ~lad_thick] = 3

        # Save the new THICK version
        nib.save(nib.Nifti1Image(thick_canvas, img.affine), final_save_path)
        print(f"   💾 Saved beautiful 3D view to: ICC_Patient_Seg_{pid}.nii.gz")

    except Exception as e:
        print(f"   🚨 ERROR inflating Patient {pid}: {str(e)}")

    # The RAM Vacuum
    try:
        del data, rca_thick, lad_thick, lcx_thick, thick_canvas
    except NameError:
        pass
    gc.collect()

print("\n🎉 ALL 3-CLASS TREES THICKENED! Ready for ITK-SNAP Inspection.")

🔍 Scanning for V5 3-class classified trees...
📊 Found 20 patients to inflate.

🎈 Inflating Patient 0169...
   -> Dilating RCA (Label 1)...
   -> Dilating LAD (Label 2)...
   -> Dilating LCx (Label 3)...
   💾 Saved beautiful 3D view to: THICK_full_tree_0169.nii.gz

🎈 Inflating Patient 0170...
   -> Dilating RCA (Label 1)...
   -> Dilating LAD (Label 2)...
   -> Dilating LCx (Label 3)...
   💾 Saved beautiful 3D view to: THICK_full_tree_0170.nii.gz

🎈 Inflating Patient 0171...
   -> Dilating RCA (Label 1)...
   -> Dilating LAD (Label 2)...
   -> Dilating LCx (Label 3)...
   💾 Saved beautiful 3D view to: THICK_full_tree_0171.nii.gz

🎈 Inflating Patient 0172...
   -> Dilating RCA (Label 1)...
   -> Dilating LAD (Label 2)...
   -> Dilating LCx (Label 3)...
   💾 Saved beautiful 3D view to: THICK_full_tree_0172.nii.gz

🎈 Inflating Patient 0173...
   -> Dilating RCA (Label 1)...
   -> Dilating LAD (Label 2)...
   -> Dilating LCx (Label 3)...
   💾 Saved beautiful 3D view to: THICK_full_tree_0173